v62 
- convert codebase for RL agent  
- headless scorer
- cross-sectional normalization
- discovery composite action
- temporal vectorizer

v61 
- **Verified all metric calculation with Excel** 
- Updated core.analyzer
- Replaced the `Result` pattern with exceptions and flattened the logic
- Refactored the `AlphaEngine` into a streamlined Orchestrator pattern
-  **Strict Date Logic:** No more "Time Travel" bugs.
-  **Dataclass Contracts:** No more "Magic String" typos or blind dictionaries.
-  **Exception Flow:** The `run` method is now a clean, readable story instead of a maze of `if/else` checks.
-  **Performance Workers:** Math is separated from orchestration.
- Ret_1d, explicitly turns division-by-zero results (`inf`) into `NaN`, and replace [np.inf, -np.inf] with np.nan



v60  
- Converted code from notebook to modular system.
- Fixed divide by zero warning from calculate_gain
- Added subtitle to subplots
- Added Volatility Regime plot


v59  
- Removed "nest" of if-statements in **AlphaEngine.run**
- Use **Result Pattern** to handle errors
- Change verify_analyzer_short and verify_analyzer_long gain calculation from simple return to logarithmic return
- Change calculate_gain from simple return to logarithmic return
- Remove bfill from calculate_gain to prevent backfill with future data
- Verify macro_df calculation


In [1]:
# 1. Enable Autoreload
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

def add_project_root_to_path():
    """Find notebooks_RLVR and add to sys.path."""
    current = Path.cwd()

    # Search upward for notebooks_RLVR folder
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            print(f"✓ Added to path: {path}")
            return path
        # Also check if notebooks_RLVR exists as child (for running from stocks/)
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            print(f"✓ Added to path: {candidate}")
            return candidate

    raise RuntimeError("Could not find notebooks_RLVR directory")


# Run once at notebook start
add_project_root_to_path()


# 2. Force reload cached modules (run this to refresh code changes)
modules_to_reload = [
    "core.analyzer",
    "core.auditor",
    "core.contracts",    
    "core.engine",
    "core.environment",
    "core.features",
    "core.paths",
    "core.performance",
    "core.quant",
    "core.result",
    "core.settings",
    "core.utils",
    "strategy.registry",
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]


# 3. Standard imports
import pandas as pd
import numpy as np

from IPython.display import display


# 4. Fresh imports (these will re-import from disk due to cache clearing above)
from core.quant import QuantUtils
from core.analyzer import create_walk_forward_analyzer
from core.auditor import SystemAuditor as SA
from core.contracts import FilterPack, EngineInput
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.paths import OUTPUT_DIR
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import METRIC_REGISTRY


# 5. Pandas display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.precision", 4)


# 6. Load data path from .env
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")

# # 7. Instantiate engine (customize DataFrames as needed)
# master_engine = AlphaEngine(
#     df_ohlcv=df_ohlcv,
#     features_df=features_df,
#     macro_df=macro_df,
#     df_close_wide=df_close_wide,
#     df_atrp_wide=df_atrp_wide,
#     df_trp_wide=df_trp_wide,
# )

✓ Added to path: c:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR
NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


In [2]:
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
print(f"df_indices:|n{df_indices}")

df_indices:|n                   Adj Open  Adj High  Adj Low  Adj Close  Volume
Ticker Date                                                      
^BSESN 1997-07-01   4263.11   4301.77  4247.66    4300.86       0
       1997-07-02   4302.96   4395.31  4295.40    4333.90       0
       1997-07-03   4335.79   4393.29  4299.97    4323.46       0
       1997-07-04   4332.70   4347.59  4300.58    4323.82       0
       1997-07-07   4326.81   4391.01  4289.49    4291.45       0
...                     ...       ...      ...        ...     ...
^VIX3M 2026-03-12     26.27     26.99    25.80      26.95       0
       2026-03-13     25.92     27.34    25.51      27.28       0
       2026-03-16     25.98     25.98    24.67      24.92       0
       2026-03-17     24.26     24.58    23.87      24.33       0
       2026-03-18     25.00     26.57    24.82      26.56       0

[136268 rows x 5 columns]


In [ ]:
_indices = df_indices.index.get_level_values(0).unique().tolist()
display(_indices)
df_indices.info()

In [3]:
print(f"Takes about 1.5 minutes")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")

Takes about 1.5 minutes


In [ ]:
# print(f"df_ohlcv.head():\n {df_ohlcv.head()}\n")
df_ohlcv.info()

In [4]:
print(f"Takes about 3 minutes to generate_features")

features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

Takes about 3 minutes to generate_features
⚡ Generating Decoupled Features (Benchmark: SPY)...


In [ ]:
print(f"features_df.info():\n{features_df.info()}\n")
print(f"features_df.index.names:\n{features_df.index.names}\n")
print(f"macro_df.info():\n{macro_df.info()}\n")
print(f"macro_df.index.names:\n{macro_df.index.names}\n")

In [5]:
# Pre-flight Automated Audit Suite
checks = [
    SA.verify_math_integrity(),
    SA.verify_ranking_integrity(),
    SA.verify_vol_alignment_integrity(),
    SA.verify_feature_engineering_integrity(),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

print("=" * 85)
# Separate verify_marco_engine output from intertwine with other outputs

checks = [
    SA.verify_macro_engine(
        df_ohlcv=df_ohlcv,
        df_indices=df_indices,
        original_macro_df=macro_df,
        settings=GLOBAL_SETTINGS,
    ),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break


--- 🛡️ Starting Feature Engineering Audit ---
⚡ Generating Decoupled Features (Benchmark: SPY)...
Audit Values:
[ nan 25.  17.5]
✅ FEATURE INTEGRITY PASSED: Wilder's ATR logic is strictly enforced.
✅ Mathematical boundaries strictly enforced.
✅ Ranking integrity strictly enforced.
✅ Reward and Risk are strictly synchronized.
✅ Wilder's ATR logic is strictly enforced.
--- Macro Verification (Benchmark: SPY) ---

Comparing verification vs original (Clip Threshold: 4.0):
✅ Mkt_Ret              | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend          | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend_Vel      | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend_Vel_Z    | PASS (Max Diff: 0.00e+00)
✅ Macro_Trend_Mom      | PASS (Max Diff: 0.00e+00)
✅ Macro_Vix_Z          | PASS (Max Diff: 0.00e+00)
✅ Macro_Vix_Ratio      | PASS (Max Diff: 0.00e+00)
✅ Macro Integrity Verified


In [6]:
print(
    "🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)"
)

# 1. Price Matrix
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)

# 2. Volatility Matrices (Unstack and Align)
# Using features_df (the first item from the tuple)
print("   - Unstacking ATRP...")
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)

print("   - Unstacking TRP...")
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# 3. Handle Data Gaps (Sanitize the Wide Matrices)
if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

# Forward fill up to the limit, then fill remaining with the "Disaster Detection" value
df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pre-computation Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)
print("   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.")

🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)
   - Unstacking ATRP...
   - Unstacking TRP...
✅ Pre-computation Complete. Tickers: 1582, Days: 16160
   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.


In [10]:
# 6. Instantiate engine (customize DataFrames as needed)
master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

In [11]:
import os
import pandas as pd
from core.environment import DiscoveryEnv

# 1. Define the Path
CACHE_PATH = os.path.join("cache", "AlphaCache_2025_V1.pkl")

# 2. Safety Check & Load
if not os.path.exists(CACHE_PATH):
    raise FileNotFoundError(f"❌ Could not find the Intelligence Cube at {CACHE_PATH}")

print(f"📂 Loading AlphaCache from {CACHE_PATH}...")
df_cube = pd.read_pickle(CACHE_PATH)

# 3. Re-inject into the Cache Object
# We pass the same lookbacks used during the 1.5-hour 'bake'
cache = AlphaCache(master_engine, lookbacks=[21, 63, 189])
cache.feature_cube = df_cube

print(f"✅ Intelligence Cube Restored: {cache.feature_cube.shape[0]} snapshots.")

# 4. Re-initialize the Discovery Environment
# This is our 'HFT-Speed' Training Gym
env = DiscoveryEnv(engine=master_engine, cache=cache, holding_period=5)

# 5. Heartbeat Verification
test_date = pd.Timestamp("2025-01-02")
obs = env.reset(start_date=test_date)
print(f"💓 System Heartbeat: Success. Current Date: {obs['date'].date()}")

📂 Loading AlphaCache from cache\AlphaCache_2025_V1.pkl...
✅ Intelligence Cube Restored: 293623 snapshots.
💓 System Heartbeat: Success. Current Date: 2025-01-02


In [ ]:
# 1. Setup Parameters
test_date = pd.Timestamp("2025-01-02")
target_metric_full = "21d_Sharpe (ATRP)"  # The Cache Name
target_metric_base = "Sharpe (ATRP)"  # The Engine Name
benchmark_ticker = "SPY"


print(f"🕵️ Starting Deep Audit for {test_date.date()}...")

# --- STEP A: THE HUMAN ENGINE (RAW TRUTH) ---
# We run the engine in 'debug' mode to get the full universe ranking
engine_input = EngineInput(
    benchmark_ticker="SPY",
    mode="Cascade",
    start_date=test_date,
    lookback_period=21,
    holding_period=5,
    metric=target_metric_base,
    rank_start=1,
    rank_end=10,
    debug=True,
)
engine_out = master_engine.run(engine_input)

# Get the full universe scores before slicing (from debug data)
raw_scores_df = engine_out.debug_data["selection_audit"]
# Let's keep Ticker, Strategy_Score (Raw), and Lookback_ATRP
engine_audit = raw_scores_df[["Strategy_Score", "Lookback_ATRP"]].rename(
    columns={"Strategy_Score": "Engine_Raw_Value"}
)

# --- STEP B: THE AI CACHE (Z-SCORED VISION) ---
# Grab the specific date slice from the cache
cache_slice = cache.get_vision(test_date)
# Get the specific Z-scored column
cache_audit = cache_slice[[target_metric_full]].rename(
    columns={target_metric_full: "Cache_Z_Score"}
)

# --- STEP C: THE MERGE (THE EXCEL MOMENT) ---
# Combine both views on the Ticker index
verification_df = engine_audit.join(cache_audit, how="inner")

# Add a 'Rank' column based on the Raw Value to verify sorting
verification_df["Engine_Rank"] = verification_df["Engine_Raw_Value"].rank(
    ascending=False
)

# Add a 'Rank' column based on the Z-Score (They should be IDENTICAL)
verification_df["Cache_Rank"] = verification_df["Cache_Z_Score"].rank(ascending=False)

# --- STEP D: THE REWARD VERIFICATION ---
# Get the reward calculated by the environment
one_hot_action = np.zeros(cache.feature_cube.shape[1] + 2)
one_hot_action[cache.feature_cube.columns.get_loc(target_metric_full)] = 1.0
one_hot_action[-2], one_hot_action[-1] = -1.0, 1.0  # Offset 0, Width 10

env.reset(start_date=test_date)
_, env_reward, _, env_info = env.step(one_hot_action)

# # --- FINAL OUTPUT ---
# print(f"✅ Audit Complete. Verification File Generated.")
# print(f"Engine Reward: {engine_out.perf_metrics['p_hold_ret_ann']:.6f} (Annualized)")
# print(f"Env Log Reward: {env_reward:.6f} (5-day Log)")

# --- FINAL OUTPUT (FIXED) ---
print("\n--- PERFORMANCE AUDIT ---")
# 1. Inspect what the Engine actually calculated
print(f"Available Metric Keys: {list(engine_out.perf_metrics.keys())}")

# 2. Get the Arithmetic Return from the Engine
# Based on our AlphaEngine logic, the key is likely 'p_hold_ret'
engine_arith_ret = engine_out.perf_metrics.get("holding_p_gain", 0.0)

# 3. Calculate what the Log Reward SHOULD be based on that Engine return
expected_log_reward = np.log1p(engine_arith_ret)

print(f"Engine Raw Period Return (Arithmetic): {engine_arith_ret:.6f}")
print(f"Expected Log Reward [LN(1+R)]:      {expected_log_reward:.6f}")
print(f"Environment Log Reward (Actual):     {env_reward:.6f}")

# 4. Precision Check
diff = abs(expected_log_reward - env_reward)
if diff < 1e-10:
    print(
        "✅ REWARD PARITY: The environment reward is a perfect Log-Mirror of the engine."
    )
else:
    print(f"⚠️ REWARD DISCREPANCY: Difference of {diff:.12f}")


# Save to CSV
verification_df.to_csv("AbsoluteZero_Verification_Audit.csv")
print(f"💾 File saved as 'AbsoluteZero_Verification_Audit.csv'. Open this in Excel!")

🕵️ Starting Deep Audit for 2025-01-02...
DEBUG: 924 stocks passed filters on 2025-01-02

--- PERFORMANCE AUDIT ---
Available Metric Keys: ['full_p_gain', 'full_p_sharpe', 'full_p_sharpe_atrp', 'full_p_sharpe_trp', 'lookback_p_gain', 'lookback_p_sharpe', 'lookback_p_sharpe_atrp', 'lookback_p_sharpe_trp', 'holding_p_gain', 'holding_p_sharpe', 'holding_p_sharpe_atrp', 'holding_p_sharpe_trp', 'full_b_gain', 'full_b_sharpe', 'full_b_sharpe_atrp', 'full_b_sharpe_trp', 'lookback_b_gain', 'lookback_b_sharpe', 'lookback_b_sharpe_atrp', 'lookback_b_sharpe_trp', 'holding_b_gain', 'holding_b_sharpe', 'holding_b_sharpe_atrp', 'holding_b_sharpe_trp']
Engine Raw Period Return (Arithmetic): -0.006031
Expected Log Reward [LN(1+R)]:      -0.006049
Environment Log Reward (Actual):     -0.006031
⚠️ REWARD DISCREPANCY: Difference of 0.000018259722
💾 File saved as 'AbsoluteZero_Verification_Audit.csv'. Open this in Excel!


In [29]:
list(cache_slice.columns)

['21d_Price Gain',
 '21d_Sharpe',
 '21d_Sharpe (ATRP)',
 '21d_Sharpe (TRP)',
 '21d_Momentum (21d)',
 '21d_Info Ratio (Stdev_Alpha, 63d)',
 '21d_Consistency (WinRate 5d)',
 '21d_Oversold (-RSI)',
 '21d_Dip Buyer (Drawdown -dd_21)',
 '21d_Low Volatility (-ATRP)',
 '21d_VIX Filtered Momentum',
 '63d_Price Gain',
 '63d_Sharpe',
 '63d_Sharpe (ATRP)',
 '63d_Sharpe (TRP)',
 '63d_Momentum (21d)',
 '63d_Info Ratio (Stdev_Alpha, 63d)',
 '63d_Consistency (WinRate 5d)',
 '63d_Oversold (-RSI)',
 '63d_Dip Buyer (Drawdown -dd_21)',
 '63d_Low Volatility (-ATRP)',
 '63d_VIX Filtered Momentum',
 '189d_Price Gain',
 '189d_Sharpe',
 '189d_Sharpe (ATRP)',
 '189d_Sharpe (TRP)',
 '189d_Momentum (21d)',
 '189d_Info Ratio (Stdev_Alpha, 63d)',
 '189d_Consistency (WinRate 5d)',
 '189d_Oversold (-RSI)',
 '189d_Dip Buyer (Drawdown -dd_21)',
 '189d_Low Volatility (-ATRP)',
 '189d_VIX Filtered Momentum']

In [27]:
# # Let's keep Ticker, Strategy_Score (Raw), and Lookback_ATRP
# engine_audit = raw_scores_df[["Strategy_Score", "Lookback_ATRP"]].rename(
#     columns={"Strategy_Score": "Engine_Raw_Value"}
# )
print(f"raw_scores_df:\n{raw_scores_df.head()}\n")
print(f"engine_audit:\n{engine_audit.head()}\n")
print(f"cache_slice:\n{cache_slice}\n")
print(f"cache_slice.colums:\n{cache_slice.colums}\n")

raw_scores_df:
        Strategy_Score  Lookback_Return_Ann  Lookback_ATRP  Was_Dropped
Ticker                                                                 
A              -0.0829              -0.5013         0.0240        False
AA             -0.2054              -2.1402         0.0414        False
AAL             0.2254               2.0001         0.0352        False
AAPL            0.0591               0.2286         0.0153        False
ABBV           -0.0248              -0.1369         0.0219        False

engine_audit:
        Engine_Raw_Value  Lookback_ATRP
Ticker                                 
A                -0.0829         0.0240
AA               -0.2054         0.0414
AAL               0.2254         0.0352
AAPL              0.0591         0.0153
ABBV             -0.0248         0.0219

cache_slice:
        21d_Price Gain  21d_Sharpe  21d_Sharpe (ATRP)  21d_Sharpe (TRP)  21d_Momentum (21d)  21d_Info Ratio (Stdev_Alpha, 63d)  21d_Consistency (WinRate 5d)  21d_Oversold (

AttributeError: 'DataFrame' object has no attribute 'colums'

In [15]:
list(engine_out.perf_metrics.keys())

['full_p_gain',
 'full_p_sharpe',
 'full_p_sharpe_atrp',
 'full_p_sharpe_trp',
 'lookback_p_gain',
 'lookback_p_sharpe',
 'lookback_p_sharpe_atrp',
 'lookback_p_sharpe_trp',
 'holding_p_gain',
 'holding_p_sharpe',
 'holding_p_sharpe_atrp',
 'holding_p_sharpe_trp',
 'full_b_gain',
 'full_b_sharpe',
 'full_b_sharpe_atrp',
 'full_b_sharpe_trp',
 'lookback_b_gain',
 'lookback_b_sharpe',
 'lookback_b_sharpe_atrp',
 'lookback_b_sharpe_trp',
 'holding_b_gain',
 'holding_b_sharpe',
 'holding_b_sharpe_atrp',
 'holding_b_sharpe_trp']

In [21]:
# engine_out.perf_metrics["holding_p_gain"]
_x = engine_out.perf_metrics.get("holding_p_gain")
print(_x)

-0.006030964991705436


In [ ]:
# 1. Setup Parameters
decision_dt = pd.Timestamp("2025-12-10")
holding_pd = 5
lookback = 21
lookback_list = [21, 63, 189]
ticker = "NVDA"  # Let's test with the ticker you verified

# A. Run via current UI logic (Manual)
# (Imagine you selected 'Sharpe (ATRP)' in the dropdown)
ui_result = master_engine.run(
    EngineInput(
        mode="Rank",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)
ui_scores = ui_result.results_df["Strategy Value"]

# B. Run via new Headless Scorer
alpha_matrix = master_engine.compute_alpha_matrix(decision_dt, lookback)
headless_scores = alpha_matrix["Sharpe (ATRP)"].reindex(ui_scores.index)

# VERIFICATION
pd.testing.assert_series_equal(ui_scores, headless_scores, check_names=False)
print("✅ Step 1 Success: Headless Scorer matches UI logic perfectly.")

In [ ]:
# --- TEST 2: NORMALIZATION INTEGRITY ---
raw_matrix = master_engine.compute_alpha_matrix(decision_dt, lookback)
norm_matrix = master_engine.normalize_alpha_matrix(raw_matrix)

# Verification A: Mean should be near 0
mean_check = norm_matrix.mean().abs().max()
# Verification B: Std should be near 1
std_check = norm_matrix.std().max()

print(f"Max Mean Offset: {mean_check:.6f} (Target: < 0.01)")
print(f"Max Std: {std_check:.6f} (Target: ~ 1.0)")

if mean_check < 0.01 and 0.8 < std_check < 1.2:
    print("✅ Step 2 Success: The Alpha Matrix is now 'Agent-Ready'.")
else:
    print("❌ Step 2 Failed: Normalization drift detected.")

In [ ]:
# --- TEST 2.1: REGIME AWARENESS VERIFICATION ---
# decision_dt = pd.Timestamp("2025-12-10")
context = master_engine.compute_context_vector(decision_dt)

print("--- Agent Macro Context ---")
print(context)

# Check against your Plotly Image (Dec 10 values):
# Trend (Green line): Should be ~10% (0.10)
# Trend Vel (Orange line): Should be slightly below 0.0
# VIX Ratio: Chart says 0.86
# VIX Z (Purple line): Should be around -1.0

print(f"\nVerification Check:")
print(f"VIX Ratio in Context: {context['Context_Vix_Ratio'] + 1:.2f} (Target: 0.86)")

In [ ]:
# # --- TEST 3 (REVISITED): VERBOSE DISCOVERY VERIFICATION ---
# # decision_dt = pd.Timestamp("2025-12-10")
# # lookback = 21

# # 1. Setup 'One-Hot' Action for Sharpe (ATRP)
# registry_keys = list(METRIC_REGISTRY.keys())
# action_weights = np.zeros(len(registry_keys))
# action_weights[registry_keys.index("Sharpe (ATRP)")] = 1.0

# # 2. Run Discovery
# discovery = master_engine.run_discovery_action(
#     decision_dt, lookback, holding_period=5, weights=action_weights
# )

# # 3. Get UI Result for verification
# ui_result = master_engine.run(
#     EngineInput(
#         mode="Ranking",
#         start_date=decision_dt,
#         lookback_period=lookback,
#         holding_period=5,
#         metric="Sharpe (ATRP)",
#         benchmark_ticker="SPY",
#     )
# )

# print(f"=== DISCOVERY ENGINE VERIFICATION (Date: {decision_dt.date()}) ===")
# print(f"Target Strategy Weights: {discovery.action_weights}")
# print(f"Selected Tickers: {discovery.selected_tickers}")
# print("-" * 50)
# print(f"Internal Discovery Score (Top 1): {discovery.metric_values.iloc[0]:.4f}")
# print(
#     f"Original UI Strategy Value (Top 1): {ui_result.results_df['Strategy Value'].iloc[0]:.4f}"
# )
# print("-" * 50)
# print(f"VERITABLE REWARD (Sharpe): {discovery.veritable_reward:.4f}")
# print(f"UI REWARD (Sharpe):        {ui_result.perf_metrics['holding_p_sharpe']:.4f}")

# if discovery.selected_tickers == ui_result.tickers:
#     print("\n✅ SELECTION MATCH: The agent and UI chose the same tickers.")

In [ ]:
# --- STEP 4.1: THE VERITABLE STANDARD PROOF ---

# # 1. Setup Parameters
# decision_dt = pd.Timestamp("2025-12-10")
# holding_pd = 5
# lookback = 21
# ticker = "SHV"  # Let's test with the ticker you verified

# 2. METHOD A: The Verified UI Engine (Compounded Daily Returns)
# This is the code you already verified with Excel
ui_result = master_engine.run(
    EngineInput(
        mode="Manual List",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=holding_pd,
        metric="Price Gain",
        manual_tickers=[ticker],
        benchmark_ticker="SPY",
    )
)
buy_date = ui_result.buy_date
end_date = ui_result.holding_end_date
ui_gain = ui_result.perf_metrics["holding_p_gain"]

# 3. METHOD B: The New Vectorized Shifter (Price Ratio)
# We calculate the log-gain directly from the price matrix
# log(P_end / P_start)
price_start = master_engine.df_close.loc[buy_date, ticker]
price_end = master_engine.df_close.loc[end_date, ticker]
vectorized_gain = np.log(price_end / price_start)

# 4. FINAL COMPARISON
print(f"=== VERITABLE STANDARD PROOF ({ticker}) ===")
print(f"Buy Date: {buy_date.date()} | End Date: {end_date.date()}")
print(f"Price at Buy: {price_start:.4f}")
print(f"Price at End: {price_end:.4f}")
print("-" * 40)
print(f"Method A (UI Engine Gain):   {ui_gain:.8f}")
print(f"Method B (Shifted Matrix):   {vectorized_gain:.8f}")
print("-" * 40)

diff = abs(ui_gain - vectorized_gain)
if diff < 1e-10:
    print("✅ VERIFICATION SUCCESS: Both methods are mathematically identical.")
    print("The Vectorized 'Time Machine' is now safe to use for RL Training.")
else:
    print(f"❌ VERIFICATION FAILED: Difference of {diff:.10f} detected.")

#

In [ ]:
# --- STEP 4.2: VECTORIZED DISCOVERY VERIFICATION ---

# A. Initialize the Time Machine (Do this once)
master_engine.precompute_reward_matrix(holding_period=5)

# decision_dt = pd.Timestamp("2025-12-10")
# lookback = 21

# B. Setup Action (Sharpe ATRP)
registry_keys = list(METRIC_REGISTRY.keys())
weights = np.zeros(len(registry_keys))
weights[registry_keys.index("Sharpe (ATRP)")] = 1.0

# C. Run Discovery (Using the NEW Vectorized functions)
# CHANGE: Only one variable on the left side
discovery = master_engine.run_discovery_action(
    decision_dt, lookback, holding_pd, weights=weights
)

# Access the matrix via the object property
raw_matrix = discovery.raw_alpha_matrix

print(f"SHV Raw Sharpe: {raw_matrix.loc['SHV', 'Sharpe (ATRP)']:.4f}")


# D. Get UI Result for the "Gold Standard"
ui_result = master_engine.run(
    EngineInput(
        mode="Ranking",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)

print(f"=== VECTORIZED ENGINE VERIFICATION (Date: {decision_dt.date()}) ===")
print(
    f"Selected Tickers: {discovery.selected_tickers[:3]}... (Match UI: {discovery.selected_tickers == ui_result.tickers})"
)
print("-" * 60)

# This confirms the 0.8381 'Signal' is preserved
print(f"SIGNAL CHECK (Lookback):")
print(f"  Discovery Score (Top 1):    {discovery.metric_values.iloc[0]:.4f}")
print(
    f"  Original UI Strategy Val:   {ui_result.results_df['Strategy Value'].iloc[0]:.4f}"
)

print("-" * 60)

# This confirms the 'Reward' matches the price action
# Note: UI Reward is Sharpe (28.29), Discovery Reward is now Total Return for RL efficiency
ui_return = ui_result.perf_metrics["holding_p_gain"]

print(f"REWARD CHECK (Holding Period Return):")
print(f"  Vectorized Reward:          {discovery.veritable_reward:.8f}")
print(f"  UI Verified Gain:           {ui_return:.8f}")

if abs(discovery.veritable_reward - ui_return) < 1e-7:
    print("\n✅ VERITABLE MATCH: The Time Machine matches the UI reality.")
    print("The Engine is now ready for High-Frequency Training.")

In [ ]:
# --- ENSEMBLE VERIFICATION TEST ---
# decision_dt = pd.Timestamp("2025-12-10")
# lookback_list = [21, 63, 189]

# 1. Generate the Ensemble
ensemble_vision = master_engine.compute_alpha_ensemble(decision_dt, lookback_list)

print(f"=== ENSEMBLE VISION SUMMARY (Date: {decision_dt.date()}) ===")
print(f"Total Features per Ticker: {ensemble_vision.shape[1]}")
print(f"Resolutions: {lookback_list}")
print("-" * 50)

# 2. Verify Component Integrity
# Let's check NVDA's Sharpe (ATRP) across the resolutions
nvda_stats = ensemble_vision.loc["NVDA"]

# Grab the 21d result to check against our 0.8410 benchmark
# Note: This will be the Z-scored/Clipped value, not raw 0.8410
val_21d = nvda_stats.get("21d_Sharpe (ATRP)")
val_189d = nvda_stats.get("189d_Sharpe (ATRP)")

print(f"NVDA 21d Sharpe (Z-Score):  {val_21d:.4f}")
print(f"NVDA 189d Sharpe (Z-Score): {val_189d:.4f}")

# 3. Check for Data Integrity
# Ensure we don't have all-NaN columns or duplicate data
if ensemble_vision.isna().sum().sum() == 0:
    print("\n✅ DATA INTEGRITY: Ensemble is fully populated (No NaNs).")

if val_21d != val_189d:
    print("✅ TEMPORAL DIFFERENTIATION: 21d and 189d signals are distinct.")
else:
    print("❌ ERROR: Temporal signals are identical (Slicing error?)")

# Display a sample of the 'Alpha Tensor' for one ticker
print("\nSample Feature Vector (NVDA):")
print(nvda_stats.head(10))

In [ ]:
# Check if the DYNAMIC metrics differ across resolutions
print(f"NVDA 21d Price Gain (Z):  {ensemble_vision.loc['NVDA', '21d_Price Gain']:.4f}")
print(f"NVDA 189d Price Gain (Z): {ensemble_vision.loc['NVDA', '189d_Price Gain']:.4f}")

if (
    ensemble_vision.loc["NVDA", "21d_Price Gain"]
    != ensemble_vision.loc["NVDA", "189d_Price Gain"]
):
    print(
        "\n✅ SLICING VERIFIED: Dynamic window metrics are correctly calculating different time horizons."
    )

In [ ]:
raw_matrix.loc["NVDA"]

In [ ]:
one_year_ago = decision_dt - pd.DateOffset(years=1)
one_year_ago

In [ ]:
# 1. Initialize Cache for a specific window
cache = AlphaCache(master_engine, lookbacks=lookback_list)

# We start in 2024 so the Agent has a year of "warm up" before 2025
one_year_ago = decision_dt - pd.DateOffset(years=1)
cache.build(start_date=one_year_ago)

# 2. Initialize the Optimized Environment
env = DiscoveryEnv(engine=master_engine, cache=cache, holding_period=holding_pd)

# 3. Benchmark Step Speed
import time

obs = env.reset(start_date=decision_dt)
action_size = (3 * 11) + 2
random_action = np.random.uniform(-1, 1, size=action_size)

print("\n⏱️ Starting High-Frequency Benchmark...")
start_time = time.time()
n_steps = 100

for i in range(n_steps):
    obs, reward, done, info = env.step(random_action)
    if done:
        env.reset()

end_time = time.time()
total_time = end_time - start_time

avg_step = total_time / n_steps

print(f"✅ Finished {n_steps} steps in {total_time:.4f} seconds.")
print(f"🚀 Average Step Time: {avg_step:.6f} seconds.")
print(
    f"📈 Projected Time for 1,000,000 steps: { (avg_step * 1_000_000) / 3600:.2f} hours (Previously: 9,700 hours)"
)

In [ ]:
from pathlib import Path

# 1. Define your pathing structure
# This ensures it works whether you are on a Local PC or Google Colab
cache_dir = Path("cache")
cache_file = cache_dir / "AlphaCache_2025_V1.pkl"

# 2. Safety First: Create the directory if it doesn't exist
# parents=True means it will create the whole path (e.g., data/discovery/cache)
# exist_ok=True means it won't crash if the folder is already there
cache_dir.mkdir(parents=True, exist_ok=True)

# 3. Save the 'Feature Cube'
# We use pickle because it preserves the Pandas MultiIndex and dtypes perfectly
try:
    cache.feature_cube.to_pickle(cache_file)
    print(f"✅ Persistence Successful!")
    print(f"💾 Path: {cache_file.absolute()}")
    print(f"📊 Cube Size: {cache.feature_cube.memory_usage().sum() / 1e6:.2f} MB")
except Exception as e:
    print(f"❌ Persistence Failed: {e}")

# 4. Quick Verify (The Trust Check)
# Ensure we can read it back immediately
import pandas as pd

test_load = pd.read_pickle(cache_file)
assert test_load.shape == cache.feature_cube.shape
print("🧪 Load-Back Test: PASSED. The data is safe.")

In [ ]:
# Diagnostic: List all 33 feature names
print("Current AlphaCache Feature Names:")
for i, col in enumerate(cache.feature_cube.columns):
    print(f"{i:02d}: {col}")

In [ ]:
# # This logic doesn't care if there's a space or an underscore!
# target_search = ["21d", "Sharpe", "ATRP"]
# matching_cols = [
#     c for c in cache.feature_cube.columns if all(word in c for word in target_search)
# ]

In [ ]:
# Re-initialize the env with the new class definition
env = DiscoveryEnv(master_engine, cache, holding_period=5)

# --- ONE-HOT PARITY TEST ---
# Search for '21d' and 'Sharpe' and '(ATRP)'
target_search = ["21d", "Sharpe", "(ATRP)"]
matching_cols = [
    c for c in cache.feature_cube.columns if all(word in c for word in target_search)
]

if not matching_cols:
    raise ValueError(
        f"❌ Search failed. Cols are: {cache.feature_cube.columns.tolist()[:5]}..."
    )

target_metric = matching_cols[0]
metric_idx = cache.feature_cube.columns.get_loc(target_metric)

# Build the One-Hot Action
action_size = cache.feature_cube.shape[1] + 2
one_hot_action = np.zeros(action_size)
one_hot_action[metric_idx] = 1.0  # Turn on ONLY this metric
one_hot_action[-2] = -1.0  # Offset = 0
one_hot_action[-1] = 1.0  # Width = 10 (Max)

# Execute Step
test_date = pd.Timestamp("2025-01-02")
env.reset(start_date=test_date)
_, _, _, info = env.step(one_hot_action)
env_tickers = info["tickers"]

# Run AlphaEngine Comparison
# Note: we pass the actual column name to see if the registry recognizes it
old_input = EngineInput(
    mode="Cascade",
    start_date=test_date,
    lookback_period=21,
    holding_period=5,
    metric="Sharpe (ATRP)",  # Standardize based on your column names
    rank_start=1,
    rank_end=10,
    benchmark_ticker="SPY",
)
old_output = master_engine.run(old_input)
engine_tickers = old_output.tickers

print(f"\n--- PARITY TEST: {target_metric} ---")
print(f"Environment: {env_tickers}")
print(f"AlphaEngine: {engine_tickers}")

if set(env_tickers) == set(engine_tickers):
    print("✅ PARITY ACHIEVED!")
else:
    print(
        "❌ Content mismatch. Check if AlphaEngine 'metric' name matches the feature."
    )

In [ ]:
import os
import pandas as pd
from core.environment import DiscoveryEnv

# 1. Define the Path
CACHE_PATH = os.path.join("cache", "AlphaCache_2025_V1.pkl")

# 2. Safety Check & Load
if not os.path.exists(CACHE_PATH):
    raise FileNotFoundError(f"❌ Could not find the Intelligence Cube at {CACHE_PATH}")

print(f"📂 Loading AlphaCache from {CACHE_PATH}...")
df_cube = pd.read_pickle(CACHE_PATH)

# 3. Re-inject into the Cache Object
# We pass the same lookbacks used during the 1.5-hour 'bake'
cache = AlphaCache(master_engine, lookbacks=[21, 63, 189])
cache.feature_cube = df_cube

print(f"✅ Intelligence Cube Restored: {cache.feature_cube.shape[0]} snapshots.")

# 4. Re-initialize the Discovery Environment
# This is our 'HFT-Speed' Training Gym
env = DiscoveryEnv(engine=master_engine, cache=cache, holding_period=5)

# 5. Heartbeat Verification
test_date = pd.Timestamp("2025-01-02")
obs = env.reset(start_date=test_date)
print(f"💓 System Heartbeat: Success. Current Date: {obs['date'].date()}")

In [ ]:
# --- STEP 5: OPTIMIZED DISCOVERY WALK ---
from core.environment import DiscoveryEnv

# 1. Initialize Gym using the CACHE (The high-speed version)
# Note: We don't pass 'lookbacks' here anymore because they are inside the cache!
env = DiscoveryEnv(
    engine=master_engine, cache=cache, holding_period=5  # <--- This is the key change
)

# 2. Define the 'Discovery Loop'
obs = env.reset(start_date=pd.Timestamp("2025-01-02"))
done = False
total_steps = 0

print(f"🚀 Starting High-Speed Discovery Walk from {obs['date'].date()}...")

# Action Size: (Lookbacks * 11 Metrics) + 2 Rank Params
# Our cache has 33 features (3 lookbacks * 11 metrics)
action_size = cache.feature_cube.shape[1] + 2

while not done:
    # Agent outputting random weights
    random_action = np.random.uniform(-1, 1, size=action_size)

    # This call is now 10,000x faster!
    obs, reward, done, info = env.step(random_action)

    if total_steps % 10 == 0:
        print(
            f"Step {total_steps:03d} | Date: {info['date'].date()} | Reward: {reward:+.4f} | Tickers: {len(info.get('tickers', []))}"
        )

    total_steps += 1

print("-" * 50)
final_return = (env.equity_curve[-1] - 1) * 100
print(f"✅ Discovery Walk Complete. Total Steps: {total_steps}")
print(f"Final Agent Equity: {env.equity_curve[-1]:.4f} ({final_return:+.2f}%)")

In [ ]:
print(f"action_size:\n{action_size}\n")
print(f"random_action :\n{random_action }\n")
print(f"obs:\n{obs}\n")
print(f"reward:\n{reward}\n")
print(f"done:\n{done}\n")
print(f"info:\n{info}\n")
print(f"env.equity_curve:\n{env.equity_curve}\n")

In [ ]:
# 1. Pick a random sample from the cache
sample_idx = 100
sample_date = cache.feature_cube.index.get_level_values("Date")[sample_idx]
sample_ticker = cache.feature_cube.index.get_level_values("Ticker")[sample_idx]

# 2. Get the value from the Cache
cached_val = cache.feature_cube.loc[(sample_date, sample_ticker)].iloc[0]

# 3. Get the value from the Engine (The slow way)
# We calculate just for this one date
engine_ensemble = master_engine.compute_alpha_ensemble(sample_date, [21, 63, 252])
engine_val = engine_ensemble.loc[sample_ticker].iloc[0]

print(f"Verification for {sample_ticker} on {sample_date.date()}:")
print(f"  Cache Value:  {cached_val:.6f}")
print(f"  Engine Value: {engine_val:.6f}")
print(f"  Difference:   {abs(cached_val - engine_val):.12f}")

assert np.isclose(cached_val, engine_val, atol=1e-8), "❌ Cache Integrity Failed!"
print("✅ Cache Integrity Verified. The fast data is identical to the trusted data.")

In [ ]:
import time

# 1. Setup
lookbacks = [21, 63, 252]
cache = AlphaCache(master_engine, lookbacks)
cache.build()  # This takes a minute, but only runs ONCE

env = DiscoveryEnv(master_engine, cache)

# 2. Benchmark the Step
obs = env.reset()
action = np.random.uniform(-1, 1, size=(len(lookbacks) * 11 + 2))

start_time = time.time()
for _ in range(100):
    _, _, done, _ = env.step(action)
    if done:
        env.reset()
end_time = time.time()

avg_step_time = (end_time - start_time) / 100
print(f"⏱️ New Average Step Time: {avg_step_time:.6f} seconds")
print(f"🚀 Speedup Factor: {35 / avg_step_time:.0f}x")

In [ ]:
obs

In [ ]:
discovery

In [ ]:
# ui_result.perf_metrics
# registry_keys
# sharpe_atrp_idx
action_weights

In [ ]:
macro_df.loc[decision_dt]

In [ ]:
# universe_subset=None means "Scan the whole market"
analyzer1, stage1_pack = create_walk_forward_analyzer(
    master_engine, universe_subset=None
)

print("🚀 Ready for Stage 1: Run Simulation for first filter.")
analyzer1.show()

In [ ]:
# tick_price_252 = analyzer1.last_run.tickers
# tick_price_189 = analyzer1.last_run.tickers
# tick_price_126 = analyzer1.last_run.tickers
# tick_price_63 = analyzer1.last_run.tickers
# tick_price_42 = analyzer1.last_run.tickers
# tick_price_21 = analyzer1.last_run.tickers
# tick_price_10 = analyzer1.last_run.tickers

In [ ]:
# tick_sharpe_atrp_252 = analyzer1.last_run.tickers
# tick_sharpe_atrp_189 = analyzer1.last_run.tickers
# tick_sharpe_atrp_126 = analyzer1.last_run.tickers
# tick_sharpe_atrp_63 = analyzer1.last_run.tickers
# tick_sharpe_atrp_42 = analyzer1.last_run.tickers
# tick_sharpe_atrp_21 = analyzer1.last_run.tickers
# tick_sharpe_atrp_10 = analyzer1.last_run.tickers

In [ ]:
union, intersection = set_operations(
    tick_sharpe_atrp_252,
    tick_sharpe_atrp_189,
    tick_sharpe_atrp_126,
    tick_sharpe_atrp_63,
    tick_sharpe_atrp_42,
    tick_sharpe_atrp_21,
    tick_sharpe_atrp_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_sharpe_atrp = intersection
# int_252_189_126 = intersection

In [ ]:
print(list_to_string(int_sharpe_atrp))

In [ ]:
def list_to_string(items, separator=", ", last_separator=None):
    """
    Convert list to string with customizable separators

    Parameters:
    -----------
    items : list of strings
    separator : str, default ', '
        Separator between items
    last_separator : str, optional
        Special separator for last item (e.g., ' and ', ' or ')

    Returns:
    --------
    str : Formatted string

    Examples:
    ---------
    list_to_string(['a', 'b', 'c'])              # "a, b, c"
    list_to_string(['a', 'b', 'c'], ' | ')        # "a | b | c"
    list_to_string(['a', 'b', 'c'], ', ', ' and ')  # "a, b and c"
    """
    if not items:
        return ""

    if len(items) == 1:
        return str(items[0])

    if last_separator and len(items) > 1:
        return separator.join(items[:-1]) + last_separator + items[-1]

    return separator.join(str(item) for item in items)


# Usage examples
print(list_to_string(["a", "b", "c"]))  # a, b, c
print(list_to_string(["a", "b", "c"], " | "))  # a | b | c
print(list_to_string(["a", "b", "c"], ", ", " and "))  # a, b and c
print(list_to_string(["a", "b", "c"], ", ", ", and "))  # a, b, and c
print(list_to_string(["apple", "banana"], ", ", " and "))  # apple and banana

In [ ]:
union, intersection = set_operations(
    tick_price_252,
    tick_price_189,
    tick_price_126,
    tick_price_63,
    tick_price_42,
    tick_price_21,
    tick_price_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_price = intersection
# int_252_189_126 = intersection

In [ ]:
union, intersection = set_operations(
    int_sharpe_atrp,
    int_price,
)
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_shp_atrp_price = intersection

In [ ]:
def set_operations(*lists):
    """
    Find sorted union, intersection, and elements unique to first list

    Parameters:
    -----------
    *lists : variable number of lists of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection, unique_to_first_list)
        - union: all unique elements from all lists
        - intersection: common elements in ALL lists
        - unique_to_first: elements only in first list, not in any other list

    Examples:
    ---------
    union, intersection, unique_first = set_operations(list1, list2)
    union, intersection, unique_first = set_operations(list1, list2, list3, list4)
    """

    if not lists:
        return [], [], []

    # Convert each list to a set
    sets = [set(lst) for lst in lists]
    first_set = sets[0]
    other_sets = sets[1:] if len(sets) > 1 else []

    # Union: all unique elements from all lists
    union_set = set().union(*sets)
    union = sorted(union_set)

    # Intersection: common elements in ALL lists
    intersection_set = first_set.intersection(*other_sets) if other_sets else first_set
    intersection = sorted(intersection_set)

    # Unique to first list: in first_set but NOT in any other set
    # Using difference: first_set - (union of all other sets)
    if other_sets:
        all_others = set().union(*other_sets)
        unique_to_first_set = (
            first_set - all_others
        )  # or first_set.difference(all_others)
    else:
        unique_to_first_set = (
            first_set  # If only one list, all elements are "unique" to it
        )

    unique_to_first = sorted(unique_to_first_set)

    return union, intersection, unique_to_first


#

In [ ]:
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage1_pack = create_walk_forward_analyzer(
    master_engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
###############################
###############################

In [ ]:
my_analyzer = analyzer1

my_res = SU.visualize_analyzer_structure(my_analyzer)

In [ ]:
def set_operations(list1, list2):
    """
    Find sorted union and intersection of two lists of strings

    Parameters:
    -----------
    list1, list2 : list of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection)
    """

    # Convert to sets for operations
    set1 = set(list1)
    set2 = set(list2)

    # Union: all unique elements from both lists
    union = sorted(set1 | set2)  # or set1.union(set2)

    # Intersection: common elements in both lists
    intersection = sorted(set1 & set2)  # or set1.intersection(set2)

    return union, intersection


# Example usage
list_a = ["apple", "banana", "cherry", "date", "elderberry"]
list_b = ["banana", "date", "fig", "grape", "apple"]

union, intersection = set_operations(list_a, list_b)

print(f"List 1: {list_a}")
print(f"List 2: {list_b}")
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

In [ ]:
union, intersection = set_operations(list_a, list_b)

In [ ]:
tick_sharpe_trp_252 = SU.peek(4, my_res)

In [ ]:
analyzer1.last_run.tickers
analyzer1.last_run.start_date
analyzer1.last_run.holding_end_date

In [ ]:
# 3. Post-flight Reconciliation
audit = SA.verify_analyzer_short(my_analyzer)
if not audit.ok:
    print(f"🚨 ALERT: {audit.msg}")
    # You could trigger a notification or log the failure here

In [ ]:
audit = SA.verify_analyzer_long(my_analyzer, n_tickers=5)

In [ ]:
# Takes 4 seconds to run, checks selected tickers from analyzer1
SA.audit_feature_engineering_integrity(my_analyzer, mode="last_run")

### Audit Every Tickers in features_df, Takes 3 minutes 

In [ ]:
# # Takes 3 minutes to run, checks every tickers ~1580 tickers
# SA.audit_feature_engineering_integrity(my_analyzer, mode="system")

### Export Ticker's OHLCV and Features

In [ ]:
f_name_excel = OUTPUT_DIR / "Audit_Verification_Report.xlsx"

SU.export_audit_to_excel(audit_pack=my_analyzer.last_run, filename=f_name_excel)

In [ ]:
f_name_csv = OUTPUT_DIR / "all_tickers_data_stacked.csv"

# Single call replaces your 3 cells
file_path = SU.export_last_run_tickers_data_to_csv(
    analyzer=my_analyzer,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename=f_name_csv,
)

In [ ]:
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=my_analyzer.last_run.tickers,
    date_start=my_analyzer.last_run.start_date,
    date_end=analyzer1.last_run.holding_end_date,
    verbose=False,
)

In [ ]:
_tic = "NVDA"
_start_date = "2025-03-12"
_end_date = "2026-03-12"
print(features_df.loc[_tic][_start_date:_end_date])
# features_df.loc[_tic][_start_date:_end_date][["Ret_1d", "Consistency"]]

In [ ]:
result = SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=[_tic],
    date_start=_start_date,
    date_end=_end_date,
    verbose=False,
)
print(result[_tic])